# Multilayer protein neural networks

Simulations supporting Figures 5–6: decision boundaries and uncertainty regions in two network architectures under shared translational resources.


In [1]:
# Colab setup
import os, sys, subprocess
if "google.colab" in sys.modules:
  cmd = "pip install --upgrade biocircuits bokeh-catplot watermark blackcellmagic intersect"
  process = subprocess.Popen(cmd.split(), stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  stdout, stderr = process.communicate()
# ------

import scipy.integrate
import biocircuits
import numba

import numpy as np
import matplotlib.pyplot as plt

import bokeh.io
import bokeh.plotting
import bokeh.palettes
from bokeh.models import LinearColorMapper, ColorBar
from bokeh.models import Range1d
from bokeh.io import export_svgs

bokeh.io.output_notebook()

Loading BokehJS ...

In [2]:
#@title Auxiliary functions

# Function to do the 2D projection of decision boundaries with normalization
def contourf(p, x, y, z, title=None, palette="Spectral11", normal = True):
    """Make a filled contour plot given x, y, z data given in 2D arrays."""

    # Normalize the z values
    if normal:
      z_min = z.min()
      z_max = z.max()
      z_normalized = (z - z_min) / (z_max - z_min)  # Normalize to range [0, 1]
    else:
      z_normalized = z

    # Add zero padding at the boundaries for better visualization
    N = z_normalized.shape[1]
    z0 = np.c_[z_normalized, np.zeros(N)]
    z0[-1, -1] = 1.  # Ensure the padding contains a value within range

    # Plot the normalized values
    p.image(
        image=[z0],
        x=x.min(),
        y=y.min(),
        dw=(x.max() - x.min()) * (1 + 1 / N),
        dh=x.max() - x.min(),
        palette=palette,
        alpha=0.8,
        level="underlay",
    )

    # Color mapping based on the normalized z0 values
    color = LinearColorMapper(palette=palette, low=z0.min(), high=z0.max(),)
    cb = ColorBar(color_mapper=color, location=(0, 0), width=10,
                  major_label_text_font_size="12pt")
    p.add_layout(cb, 'right')

    return ()

def numerical_decision_boundary(p, x1, x2, output, level=0.05,color='black', dash = 'dashed'):

    # Extract zero-level contour from output for decision boundary
    X1, X2 = np.meshgrid(x1, x2)
    contours = plt.contour(X1, X2, output, levels=[level])
    plt.close()  # prevent displaying the plot

    for segs in contours.allsegs:  # list of contour levels (only 1 level here)
        for seg in segs:           # each seg = one continuous path
            p.line(seg[:, 0], seg[:, 1], line_width=2, color=color, line_dash=dash)

## Architecture A: split-protein network with a protein-complex output


In [3]:
def Seq_TL_ODEs_Split(X, t, pm):

    # State vector
    m3,r3,y3, m1,m2,r1,r2,y1,y2,c1, m4,m5,m6,r4,r5,r6,y4,y5,c2, m7,m8,r7,r8,y6,y7,c3 = X

    # Available ribosome
    r = pm.rtot - (r1 + r2 + r3 + r4 + r5 + r6 + r7 + r8)

    return [
        # ------------- Endogenous demand -------------------- #
        pm.ap_03*pm.x3 - pm.p*m3 - pm.d*m3 - pm.a3*m3*r + pm.d3*r3 + pm.v3*r3,
        pm.a3*m3*r - pm.d3*r3 - pm.v3*r3 - pm.d*r3 - pm.p*r3,
        pm.v3*r3 - pm.d*y3,

        # ------------------ Node 1 -------------------------- #
        pm.ap_11*pm.x1 - pm.p*m1 - pm.d*m1 - pm.a1*m1*r + pm.d1*r1 + pm.v1*r1,
        pm.ap_12*pm.x2 - pm.p*m2 - pm.d*m2 - pm.a2*m2*r + pm.d2*r2 + pm.v2*r2,
        pm.a1*m1*r - pm.d1*r1 - pm.v1*r1 - pm.d*r1 - pm.p*r1,
        pm.a2*m2*r - pm.d2*r2 - pm.v2*r2 - pm.d*r2 - pm.p*r2,
        pm.v1*r1 - pm.d*y1 - pm.b1*y1*y2 + pm.u1*c1,
        pm.v2*r2 - pm.d*y2 - pm.b1*y1*y2 + pm.u1*c1,
        pm.b1*y1*y2 - pm.u1*c1 - pm.d*c1,

        # ------------------ Node 2 -------------------------- #
        pm.ap_21*pm.x1 - pm.p*m4 - pm.d*m4 - pm.a4*m4*r + pm.d4*r4 + pm.v4*r4,
        pm.ap_22*pm.x2 - pm.p*m5 - pm.d*m5 - pm.a5*m5*r + pm.d5*r5 + pm.v5*r5,
        pm.ap_20*pm.x0 - pm.p*m6 - pm.d*m6 - pm.a6*m6*r + pm.d6*r6 + pm.v6*r6,
        pm.a4*m4*r - pm.d4*r4 - pm.v4*r4 - pm.d*r4 - pm.p*r4,
        pm.a5*m5*r - pm.d5*r5 - pm.v5*r5 - pm.d*r5 - pm.p*r5,
        pm.a6*m6*r - pm.d6*r6 - pm.v6*r6 - pm.d*r6 - pm.p*r6,
        pm.v4*r4 + pm.v5*r5 - pm.d*y4 - pm.b2*y4*y5 + pm.u2*c2,
        pm.v6*r6 - pm.d*y5 - pm.b2*y4*y5 + pm.u2*c2,
        pm.b2*y4*y5 - pm.u2*c2 - pm.d*c2,

        # ------------------ Output (Node 3) -------------------- #
        pm.ap_31*y1 - pm.p*m7 - pm.d*m7 - pm.a7*m7*r + pm.d7*r7 + pm.v7*r7,
        pm.ap_32*y4 - pm.p*m8 - pm.d*m8 - pm.a8*m8*r + pm.d8*r8 + pm.v8*r8,
        pm.a7*m7*r - pm.d7*r7 - pm.v7*r7 - pm.d*r7 - pm.p*r7,
        pm.a8*m8*r - pm.d8*r8 - pm.v8*r8 - pm.d*r8 - pm.p*r8,
        pm.v7*r7 - pm.d*y6 - pm.b3*y6*y7 + pm.u3*c3,
        pm.v8*r8 - pm.d*y7 - pm.b3*y6*y7 + pm.u3*c3,
        pm.b3*y6*y7 - pm.u3*c3 - pm.d*c3,
    ]

In [4]:
class TL_Competition_Split:
    def __init__(self):

        # Degradation/dilution rates
        self.p = 10.0    # hr⁻¹, mRNA decay
        self.d = 1.0     # hr⁻¹, dilution by growth

        # Transcription rates (μM/hr)
        self.ap_03 = 10.0   # Endogenous demand
        self.ap_11 = 10.0   # Node 1, x1-driven → y1
        self.ap_12 = 10.0   # Node 1, x2-driven → y2
        self.ap_21 = 10.0   # Node 2, x1-driven → y4
        self.ap_22 = 10.0   # Node 2, x2-driven → y4
        self.ap_20 = 10.0   # Node 2, x0-driven → y5
        self.ap_31 = 10.0   # Output, y1-driven → y6
        self.ap_32 = 10.0   # Output, y4-driven → y7

        # Ribosome–mRNA association (μM⁻¹·hr⁻¹)
        self.a1 = self.a2 = self.a3 = 100.0       # Demand, Node 1
        self.a4 = self.a5 = self.a6 = 100.0       # Node 2
        self.a7 = self.a8 = 100.0                 # Output

        # Ribosome–mRNA dissociation (hr⁻¹)
        self.d1 = self.d2 = self.d3 = 1.0
        self.d4 = self.d5 = self.d6 = 1.0
        self.d7 = self.d8 = 1.0

        # Translation rates (hr⁻¹)
        self.v1 = self.v2 = self.v3 = 1.0
        self.v4 = self.v5 = self.v6 = 1.0
        self.v7 = self.v8 = 1.0

        # Sequestration
        self.b1 = self.b2 = self.b3 = 200.0  # μM⁻¹·hr⁻¹
        self.u1 = self.u2 = self.u3 = 1.0    # hr⁻¹

        # Ribosome pool
        self.rtot = 3.0  # μM

        # Input signals
        self.x0 = self.x1 = self.x2 = self.x3 = 1.0

In [5]:
def run_split_mlp(pm, x1, x2, N, t):
    """
    Run MLP simulation over a grid of (x1, x2) inputs.
    Returns output arrays for Node 1 (y1), Node 2 (y4), and Output (c3).
    State vector (26): m3,r3,y3, m1,m2,r1,r2,y1,y2,c1,
                       m4,m5,m6,r4,r5,r6,y4,y5,c2,
                       m7,m8,r7,r8,y6,y7,c3
    pm: TL_Competition_Split
    x1, x2: input arrays
    N: grid resolution (int)
    t: time array
    """
    output1 = np.zeros((N, N))
    output2 = np.zeros((N, N))
    output3 = np.zeros((N, N))
    for i, x1i in enumerate(x1):
        for j, x2j in enumerate(x2):
            pm.x1 = x1i
            pm.x2 = x2j
            y_sol = scipy.integrate.odeint(Seq_TL_ODEs_Split, np.zeros(26), t, args=(pm,))
            output1[j, i] = y_sol[-1, 7]   # y1 — Node 1
            output2[j, i] = y_sol[-1, 16]  # y4 — Node 2
            output3[j, i] = y_sol[-1, 25]  # c3 — Output
    return output1, output2, output3


def get_boundary(output, x1, x2, level=0.05):
    """
    Extract the longest decision boundary contour at a given output level.
    output: (N, N) array of steady-state values
    x1, x2: input arrays defining the grid
    level: contour threshold
    """
    contour = plt.contour(x1, x2, output, levels=[level])
    segs = contour.allsegs[0]
    plt.close()
    if len(segs) > 0:
        return max(segs, key=lambda v: v.shape[0])
    return None


def make_envelope(pm, x1, x2, N, t, var_factors, output_idx, level=0.05):
    """
    Sweep var_factors around the nominal favored affinity K_base/v_factor
    and extract decision boundary contours for each variation.
    Returns a list of boundary vertex arrays (one per var_factor value).
    pm: TL_Competition_Split
    var_factors: e.g. [1+vr, 1.0, 1-vr], applied as K_base * vf / v_factor
    output_idx: state vector index to extract (7=y1, 16=y4, 25=c3)
    level: contour threshold for boundary extraction
    """
    boundaries = []
    for vf in var_factors:
        # Favored affinities swept around nominal K_base/v_factor
        pm.a1 = (pm.d1 + pm.v1 + pm.d + pm.p) / (K_base * vf / v_factor)  # Node 1, x1
        pm.a2 = (pm.d2 + pm.v2 + pm.d + pm.p) / K_base                     # Node 1, x2
        pm.a3 = (pm.d3 + pm.v3 + pm.d + pm.p) / K_base                     # Endogenous
        pm.a4 = (pm.d4 + pm.v4 + pm.d + pm.p) / (K_base * vf / v_factor)  # Node 2, x1
        pm.a5 = (pm.d5 + pm.v5 + pm.d + pm.p) / K_base                     # Node 2, x2
        pm.a6 = (pm.d6 + pm.v6 + pm.d + pm.p) / K_base                     # Node 2, x0
        pm.a7 = (pm.d7 + pm.v7 + pm.d + pm.p) / K_base                     # Output, y1
        pm.a8 = (pm.d8 + pm.v8 + pm.d + pm.p) / K_base                     # Output, y4

        output = np.zeros((N, N))
        for i, x1i in enumerate(x1):
            for j, x2j in enumerate(x2):
                pm.x1 = x1i
                pm.x2 = x2j
                y_sol = scipy.integrate.odeint(Seq_TL_ODEs_Split, np.zeros(26), t, args=(pm,))
                output[j, i] = y_sol[-1, output_idx]

        bnd = get_boundary(output, x1, x2, level=level)
        if bnd is not None:
            boundaries.append(bnd)
    return boundaries


def draw_envelope(plot, boundaries, color, alpha=0.4):
    """
    Draw shaded envelope between the outer (low and high variation)
    boundaries, with solid black lines marking the envelope edges.
    plot: bokeh figure
    boundaries: list of boundary vertex arrays from make_envelope
    color: fill color for the shaded region
    alpha: transparency of the shaded region
    """
    if len(boundaries) < 2:
        return

    # Orient both outer boundaries left-to-right
    lo = boundaries[0]  if boundaries[0][-1,  0] >= boundaries[0][0,  0] else boundaries[0][::-1]
    hi = boundaries[-1] if boundaries[-1][-1, 0] >= boundaries[-1][0, 0] else boundaries[-1][::-1]

    # Shaded region between outer boundaries
    xs = np.concatenate([lo[:, 0], hi[::-1, 0]])
    ys = np.concatenate([lo[:, 1], hi[::-1, 1]])
    plot.patch(xs, ys, color=color, alpha=alpha, line_alpha=0)

    # Solid black lines marking the envelope edges
    plot.line(lo[:, 0], lo[:, 1], line_color='black', line_width=2)
    plot.line(hi[:, 0], hi[:, 1], line_color='black', line_width=2)

### Figure 5A2: homogeneous affinities without endogenous demand


In [6]:
# Color palette
palette = bokeh.palettes.grey(15)[::-1]

# Simulation time (hours)
t = np.linspace(0, 10, 100)

# Resolution and input array
N = 25
x1 = np.linspace(0, 1, N)
x2 = np.linspace(0, 1, N)

# Set up figure
fig_size = [320, 225]
plots = []
for i in range(3):
    plots.append(bokeh.plotting.figure(
        width=fig_size[0], height=fig_size[1],
        x_range=bokeh.models.Range1d(x1[0], x1[-1]),
        y_range=bokeh.models.Range1d(x2[0], x2[-1])
    ))
    plots[i].axis.major_label_text_font_size = '13pt'

# Solving the ODEs
pm = TL_Competition_Split()
pm.x3 = 0.0

output1_c1, output2_c1, output3_c1 = run_split_mlp(pm, x1, x2, N, t)

# Reconstructing the heatmaps in the x1-x2 plane
contourf(plots[0], x1, x2, output1_c1, palette=palette, normal=False)
numerical_decision_boundary(plots[0], x1, x2, output1_c1, level=0.05)
contourf(plots[1], x1, x2, output2_c1, palette=palette, normal=False)
numerical_decision_boundary(plots[1], x1, x2, output2_c1, level=0.05)
contourf(plots[2], x1, x2, output3_c1, palette=palette, normal=False)
numerical_decision_boundary(plots[2], x1, x2, output1_c1, level=0.05, color='grey')
numerical_decision_boundary(plots[2], x1, x2, output2_c1, level=0.05, color='grey')
numerical_decision_boundary(plots[2], x1, x2, output3_c1, level=0.05)

# Set up export and show plots
for i in range(3):
    plots[i].output_backend = 'svg'

bokeh.io.show(bokeh.layouts.row(plots))

### Figure 5A3: homogeneous affinities with endogenous demand


In [7]:
# Color palette
palette = bokeh.palettes.grey(15)[::-1]

# Simulation time (hours)
t = np.linspace(0, 10, 100)

# Resolution and input array
N = 25
x1 = np.linspace(0, 1, N)
x2 = np.linspace(0, 1, N)

# Set up figure
fig_size = [320, 225]
plots = []
for i in range(3):
    plots.append(bokeh.plotting.figure(
        width=fig_size[0], height=fig_size[1],
        x_range=bokeh.models.Range1d(x1[0], x1[-1]),
        y_range=bokeh.models.Range1d(x2[0], x2[-1])
    ))
    plots[i].axis.major_label_text_font_size = '13pt'

# Solving the ODEs
pm = TL_Competition_Split()
pm.x3 = 1.5

output1_c1, output2_c1, output3_c1 = run_split_mlp(pm, x1, x2, N, t)

# Reconstructing the heatmaps in the x1-x2 plane
contourf(plots[0], x1, x2, output1_c1, palette=palette, normal=False)
numerical_decision_boundary(plots[0], x1, x2, output1_c1, level=0.05)
contourf(plots[1], x1, x2, output2_c1, palette=palette, normal=False)
numerical_decision_boundary(plots[1], x1, x2, output2_c1, level=0.05)
contourf(plots[2], x1, x2, output3_c1, palette=palette, normal=False)
numerical_decision_boundary(plots[2], x1, x2, output1_c1, level=0.05, color='grey')
numerical_decision_boundary(plots[2], x1, x2, output2_c1, level=0.05, color='grey')
numerical_decision_boundary(plots[2], x1, x2, output3_c1, level=0.05)

# Set up export and show plots
for i in range(3):
    plots[i].output_backend = 'svg'

bokeh.io.show(bokeh.layouts.row(plots))

### Figure 6A1: heterogeneous affinities with endogenous demand


In [8]:
# Color palette
palette = bokeh.palettes.grey(15)[::-1]

# Simulation time (hours)
t = np.linspace(0, 10, 100)

# Heterogeneous case
K_base = 1.1375e-05
v_factor = 5

pm.x3 = 1.5

# Resolution and input array
N = 25
x1 = np.linspace(0, 1, N)
x2 = np.linspace(0, 1, N)

# Set up figure
fig_size = [320, 225]
plots = []
for i in range(3):
    plots.append(bokeh.plotting.figure(
        width=fig_size[0], height=fig_size[1],
        x_range=bokeh.models.Range1d(x1[0], x1[-1]),
        y_range=bokeh.models.Range1d(x2[0], x2[-1])
    ))
    plots[i].axis.major_label_text_font_size = '13pt'

# Solving the ODEs
pm = TL_Competition_Split()

# Generating the heterogeneous case
pm.a1 = (pm.d1 + pm.v1 + pm.d + pm.p) / (K_base / v_factor)
pm.a2 = (pm.d2 + pm.v2 + pm.d + pm.p) / K_base
pm.a3 = (pm.d3 + pm.v3 + pm.d + pm.p) / K_base
pm.a4 = (pm.d4 + pm.v4 + pm.d + pm.p) / (K_base / v_factor)
pm.a5 = (pm.d5 + pm.v5 + pm.d + pm.p) / K_base
pm.a6 = (pm.d6 + pm.v6 + pm.d + pm.p) / K_base # (K_base / v_factor) #
pm.a7 = (pm.d7 + pm.v7 + pm.d + pm.p) / K_base
pm.a8 = (pm.d8 + pm.v8 + pm.d + pm.p) / K_base

output1_c1, output2_c1, output3_c1 = run_split_mlp(pm, x1, x2, N, t)

# Reconstructing the heatmaps in the x1-x2 plane
contourf(plots[0], x1, x2, output1_c1, palette=palette, normal=False)
numerical_decision_boundary(plots[0], x1, x2, output1_c1, level=0.05)
contourf(plots[1], x1, x2, output2_c1, palette=palette, normal=False)
numerical_decision_boundary(plots[1], x1, x2, output2_c1, level=0.05)
contourf(plots[2], x1, x2, output3_c1, palette=palette, normal=False)
numerical_decision_boundary(plots[2], x1, x2, output1_c1, level=0.05, color='grey')
numerical_decision_boundary(plots[2], x1, x2, output2_c1, level=0.05, color='grey')
numerical_decision_boundary(plots[2], x1, x2, output3_c1, level=0.05)

# Set up export and show plots
for i in range(3):
    plots[i].output_backend = 'svg'

bokeh.io.show(bokeh.layouts.row(plots))

### Figure 6A2: uncertainty under binding-affinity variation


In [9]:
# Split-protein model: high-output regions and uncertainty.
# Requires only the notebook imports and split-model definitions above.
import contourpy

# Explicit setup: independent of all earlier plotting/simulation cells.
t = np.linspace(0, 10, 100)
K_base = 1.1375e-05
v_factor = 5
vr = 0.5
N = 25
x1 = np.linspace(0, 1, N)
x2 = np.linspace(0, 1, N)

_split_endpoint_outputs = []
for vf in (1 + vr, 1 - vr):
    _split_pm = TL_Competition_Split()
    _split_pm.rtot = 3.0
    # Preserve the original uncertainty simulation's effective demand setting.
    _split_pm.x3 = 1.0
    _split_pm.a1 = (_split_pm.d1 + _split_pm.v1 + _split_pm.d + _split_pm.p) / (K_base * vf / v_factor)
    _split_pm.a2 = (_split_pm.d2 + _split_pm.v2 + _split_pm.d + _split_pm.p) / K_base
    _split_pm.a3 = (_split_pm.d3 + _split_pm.v3 + _split_pm.d + _split_pm.p) / K_base
    _split_pm.a4 = (_split_pm.d4 + _split_pm.v4 + _split_pm.d + _split_pm.p) / (K_base * vf / v_factor)
    _split_pm.a5 = (_split_pm.d5 + _split_pm.v5 + _split_pm.d + _split_pm.p) / K_base
    _split_pm.a6 = (_split_pm.d6 + _split_pm.v6 + _split_pm.d + _split_pm.p) / K_base
    _split_pm.a7 = (_split_pm.d7 + _split_pm.v7 + _split_pm.d + _split_pm.p) / K_base
    _split_pm.a8 = (_split_pm.d8 + _split_pm.v8 + _split_pm.d + _split_pm.p) / K_base
    _split_endpoint_outputs.append(run_split_mlp(_split_pm, x1, x2, N, t))


def _split_fill_above(plot, field, threshold, color):
    """Fill a superlevel region, preserving disconnected polygons and holes."""
    field = np.asarray(field)
    upper = float(np.max(field))
    if upper <= threshold:
        return
    upper += max(abs(upper), abs(threshold), 1.0) * 1e-6
    generator = contourpy.contour_generator(
        x=x1, y=x2, z=field, fill_type="OuterOffset",
    )
    polygons, offsets = generator.filled(threshold, upper)
    polygons_x, polygons_y = [], []
    for vertices, ring_offsets in zip(polygons, offsets):
        rings = [
            vertices[start:stop]
            for start, stop in zip(ring_offsets[:-1], ring_offsets[1:])
        ]
        polygons_x.append([ring[:, 0].tolist() for ring in rings])
        polygons_y.append([ring[:, 1].tolist() for ring in rings])
    if polygons_x:
        plot.multi_polygons(
            xs=[polygons_x], ys=[polygons_y],
            fill_color=color, fill_alpha=1.0, line_color=None,
        )


def _split_draw_boundary(plot, field, threshold):
    """Draw contour segments separately, without closing open boundaries."""
    generator = contourpy.contour_generator(
        x=x1, y=x2, z=field, line_type="Separate",
    )
    for segment in generator.lines(threshold):
        if len(segment) > 1:
            plot.line(
                segment[:, 0], segment[:, 1],
                line_color="black", line_width=2,
            )


split_region_plots = []
_split_colors = [
    ("#AEC0DC", "#E8EFF8"),
    ("#EFBEA0", "#FBEDE3"),
    ("#B7B7B7", "#ECECEC"),
]
for node_index, (band_color, light_color) in enumerate(_split_colors):
    output_lo = _split_endpoint_outputs[0][node_index]
    output_hi = _split_endpoint_outputs[1][node_index]
    plot = bokeh.plotting.figure(
        width=275, height=225,
        x_range=bokeh.models.Range1d(x1[0], x1[-1]),
        y_range=bokeh.models.Range1d(x2[0], x2[-1]),
        background_fill_color="white", output_backend="svg",
    )
    plot.axis.major_label_text_font_size = "13pt"

    # Dark: either endpoint is high. Light: both endpoints are high.
    _split_fill_above(plot, np.maximum(output_lo, output_hi), 0.05, band_color)
    _split_fill_above(plot, np.minimum(output_lo, output_hi), 0.05, light_color)
    _split_draw_boundary(plot, output_lo, 0.05)
    _split_draw_boundary(plot, output_hi, 0.05)
    split_region_plots.append(plot)

bokeh.io.show(bokeh.layouts.row(*split_region_plots))


## Architecture B: fully connected network with a free-protein output


In [10]:
class TL_Competition:
    def __init__(self):
        self.p = 10.0
        self.d = 1.0

        # ---------------- Transcription rates ---------------- #
        self.ap_03 = 10.0   # demand, x3-driven

        self.ap_11 = self.ap_12 = 10.0
        self.ap_10 = 1.0

        self.ap_21 = self.ap_22 = 10.0
        self.ap_20 = 1.0

        self.ap_31 = self.ap_32 = self.ap_30 = 10.0

        # ---------------- Ribosome–mRNA association ---------------- #
        self.a1 = self.a2 = self.a3 = self.a4 = \
        self.a5 = self.a6 = self.a7 = self.a8 = \
        self.a9 = self.a10 = self.a11 = 100.0

        # ---------------- Ribosome–mRNA dissociation ---------------- #
        self.d1 = self.d2 = self.d3 = self.d4 = \
        self.d5 = self.d6 = self.d7 = self.d8 = \
        self.d9 = self.d10 = self.d11 = 1.0

        # ---------------- Translation rates ---------------- #
        self.v1 = self.v2 = self.v3 = self.v4 = \
        self.v5 = self.v6 = self.v7 = self.v8 = \
        self.v9 = self.v10 = self.v11 = 1.0

        # ---------------- Sequestration ---------------- #
        self.b1 = self.b2 = self.b3 = 200.0
        self.u1 = self.u2 = self.u3 = 1.0

        # ---------------- Ribosome pool ---------------- #
        self.rtot = 2.0

        # ---------------- Inputs ---------------- #
        self.x0 = self.x1 = self.x2 = self.x3 = 1.0

In [11]:
def Seq_TL_ODEs(X, t, pm):
    # State vector
    (m3, r3, y3,
     m1, m2, m4, r1, r2, r4, y1, y2, c1,
     m5, m6, m8, r5, r6, r8, y4, y5, c2,
     m9, m11, m10, r9, r11, r10, y6, y7, c3) = X

    # Available ribosomes
    r = pm.rtot - (r1 + r2 + r4 + r3 +
                   r5 + r6 + r8 +
                   r9 + r11 + r10)

    return [

        # ------------- Endogenous demand -------------------- #
        pm.ap_03*pm.x3 - pm.p*m3 - pm.d*m3 - pm.a3*m3*r + pm.d3*r3 + pm.v3*r3,
        pm.a3*m3*r - pm.d3*r3 - pm.v3*r3 - pm.d*r3 - pm.p*r3,
        pm.v3*r3 - pm.d*y3,

        # ------------------ Node 1 ---------------- #
        pm.ap_11*pm.x1 - pm.p*m1 - pm.d*m1 - pm.a1*m1*r + pm.d1*r1 + pm.v1*r1,
        pm.ap_12*pm.x2 - pm.p*m2 - pm.d*m2 - pm.a2*m2*r + pm.d2*r2 + pm.v2*r2,
        pm.ap_10*pm.x0 - pm.p*m4 - pm.d*m4 - pm.a4*m4*r + pm.d4*r4 + pm.v4*r4,
        pm.a1*m1*r - pm.d1*r1 - pm.v1*r1 - pm.d*r1 - pm.p*r1,
        pm.a2*m2*r - pm.d2*r2 - pm.v2*r2 - pm.d*r2 - pm.p*r2,
        pm.a4*m4*r - pm.d4*r4 - pm.v4*r4 - pm.d*r4 - pm.p*r4,
        pm.v1*r1 - pm.d*y1 - pm.b1*y1*y2 + pm.u1*c1,
        pm.v2*r2 + pm.v4*r4 - pm.d*y2 - pm.b1*y1*y2 + pm.u1*c1,
        pm.b1*y1*y2 - pm.u1*c1 - pm.d*c1,

        # ------------------ Node 2 ---------------- #
        pm.ap_22*pm.x2 - pm.p*m5 - pm.d*m5 - pm.a5*m5*r + pm.d5*r5 + pm.v5*r5,
        pm.ap_21*pm.x1 - pm.p*m6 - pm.d*m6 - pm.a6*m6*r + pm.d6*r6 + pm.v6*r6,
        pm.ap_20*pm.x0 - pm.p*m8 - pm.d*m8 - pm.a8*m8*r + pm.d8*r8 + pm.v8*r8,
        pm.a5*m5*r - pm.d5*r5 - pm.v5*r5 - pm.d*r5 - pm.p*r5,
        pm.a6*m6*r - pm.d6*r6 - pm.v6*r6 - pm.d*r6 - pm.p*r6,
        pm.a8*m8*r - pm.d8*r8 - pm.v8*r8 - pm.d*r8 - pm.p*r8,
        pm.v5*r5 - pm.d*y4 - pm.b2*y4*y5 + pm.u2*c2,
        pm.v6*r6 + pm.v8*r8 - pm.d*y5 - pm.b2*y4*y5 + pm.u2*c2,
        pm.b2*y4*y5 - pm.u2*c2 - pm.d*c2,

        # ------------------ Output ---------------- #
        pm.ap_31*y1 - pm.p*m9 - pm.d*m9 - pm.a9*m9*r + pm.d9*r9 + pm.v9*r9,
        pm.ap_32*y4 - pm.p*m11 - pm.d*m11 - pm.a11*m11*r + pm.d11*r11 + pm.v11*r11,
        pm.ap_30*pm.x0 - pm.p*m10 - pm.d*m10 - pm.a10*m10*r + pm.d10*r10 + pm.v10*r10,
        pm.a9*m9*r - pm.d9*r9 - pm.v9*r9 - pm.d*r9 - pm.p*r9,
        pm.a11*m11*r - pm.d11*r11 - pm.v11*r11 - pm.d*r11 - pm.p*r11,
        pm.a10*m10*r - pm.d10*r10 - pm.v10*r10 - pm.d*r10 - pm.p*r10,
        pm.v9*r9 + pm.v11*r11 - pm.d*y6 - pm.b3*y6*y7 + pm.u3*c3,
        pm.v10*r10 - pm.d*y7 - pm.b3*y6*y7 + pm.u3*c3,
        pm.b3*y6*y7 - pm.u3*c3 - pm.d*c3,
    ]

In [12]:
def get_boundary(output, x1, x2, level=0.05):
    """Extract and concatenate all contour segments at a given level."""
    fig, ax = plt.subplots()
    contour = ax.contour(x1, x2, output, levels=[level])
    segs = contour.allsegs[0]
    plt.close(fig)

    if len(segs) == 0:
        return None, 0
    if len(segs) == 1:
        return segs[0], 1

    segs_sorted = sorted(segs, key=lambda v: v[0, 0])
    return np.concatenate(segs_sorted, axis=0), len(segs)


def run_mlp(pm, x1, x2, N, t):
    """Run MLP simulation over a grid of (x1, x2) inputs."""
    output1 = np.zeros((N, N))
    output2 = np.zeros((N, N))
    output3 = np.zeros((N, N))

    for i, x1i in enumerate(x1):
        for j, x2j in enumerate(x2):
            pm.x1 = x1i
            pm.x2 = x2j
            y_sol = scipy.integrate.odeint(Seq_TL_ODEs, np.zeros(30), t, args=(pm,))
            output1[j, i] = y_sol[-1, 9]   # y1
            output2[j, i] = y_sol[-1, 18]  # y4
            output3[j, i] = y_sol[-1, 27]  # c3

    return output1, output2, output3


def make_envelope(pm, x1, x2, N, t, var_factors, output_idx, level=0.05):
    """
    Run simulations for each variation factor and extract decision
    boundary contours. Returns a list of (boundary, n_segments) tuples.
    """
    a1_base  = (pm.d1  + pm.v1  + pm.d + pm.p) / K_base
    a2_base  = (pm.d2  + pm.v2  + pm.d + pm.p) / K_base
    a4_base  = (pm.d4  + pm.v4  + pm.d + pm.p) / K_base
    a5_base  = (pm.d5  + pm.v5  + pm.d + pm.p) / K_base
    a6_base  = (pm.d6  + pm.v6  + pm.d + pm.p) / K_base
    a8_base  = (pm.d8  + pm.v8  + pm.d + pm.p) / K_base
    a9_base  = (pm.d9  + pm.v9  + pm.d + pm.p) / K_base
    a10_base = (pm.d10 + pm.v10 + pm.d + pm.p) / K_base
    a11_base = (pm.d11 + pm.v11 + pm.d + pm.p) / K_base

    boundaries = []

    for vf in var_factors:
        pm.a1  = a1_base  * vf / v_factor
        pm.a6  = a6_base  * vf / v_factor
        pm.a2  = a2_base
        pm.a4  = a4_base
        pm.a5  = a5_base
        pm.a8  = a8_base
        pm.a9  = a9_base
        pm.a10 = a10_base
        pm.a11 = a11_base

        output = np.zeros((N, N))
        for i, x1i in enumerate(x1):
            for j, x2j in enumerate(x2):
                pm.x1 = x1i
                pm.x2 = x2j
                y_sol = scipy.integrate.odeint(Seq_TL_ODEs, np.zeros(30), t, args=(pm,))
                output[j, i] = y_sol[-1, output_idx]

        bnd, n_segs = get_boundary(output, x1, x2, level=level)
        print(f"var_factor={vf:.3f}: boundary {'found' if bnd is not None else 'NOT FOUND'}, segments={n_segs}")

        if bnd is not None:
            boundaries.append(bnd)

    return boundaries


def draw_envelope(plot, boundaries, color, alpha=0.4):
    """
    Draw shaded envelope between the outer boundaries,
    with solid black lines marking the envelope edges.
    """
    if len(boundaries) < 2:
        print("Warning: fewer than 2 boundaries found, cannot draw envelope.")
        return

    lo = boundaries[0]  if boundaries[0][0, 0]  <= boundaries[0][-1, 0]  else boundaries[0][::-1]
    hi = boundaries[-1] if boundaries[-1][0, 0] <= boundaries[-1][-1, 0] else boundaries[-1][::-1]

    xs = np.concatenate([lo[:, 0], hi[::-1, 0]])
    ys = np.concatenate([lo[:, 1], hi[::-1, 1]])
    plot.patch(xs, ys, color=color, alpha=alpha, line_alpha=0)
    plot.line(lo[:, 0], lo[:, 1], line_color='black', line_width=2)
    plot.line(hi[:, 0], hi[:, 1], line_color='black', line_width=2)

### Figure 5B2: homogeneous affinities without endogenous demand


In [13]:
from bokeh.models import LinearColorMapper, ColorBar, BasicTickFormatter

palette = bokeh.palettes.grey(15)[::-1]
t = np.linspace(0, 10, 100)

pm = TL_Competition()
pm.x3 = 0.0

N = 25
x1 = np.linspace(0, 1, N)
x2 = np.linspace(0, 1, N)

fig_size = [320, 225]
plots = []
for i in range(3):
    plots.append(
        bokeh.plotting.figure(
            width=fig_size[0],
            height=fig_size[1],
            x_range=bokeh.models.Range1d(x1[0], x1[-1]),
            y_range=bokeh.models.Range1d(x2[0], x2[-1])
        )
    )
    plots[i].axis.major_label_text_font_size = '13pt'

# ---------------- Run simulation ---------------- #
output1, output2, output3 = run_mlp(pm, x1, x2, N, t)

# ---------------- Node 1 ---------------- #
contourf(plots[0], x1, x2, output1, palette=palette, normal=False)
numerical_decision_boundary(plots[0], x1, x2, output1, level=0.05)

# ---------------- Node 2 ---------------- #
contourf(plots[1], x1, x2, output2, palette=palette, normal=False)
numerical_decision_boundary(plots[1], x1, x2, output2, level=0.05)

# ---------------- Output ---------------- #
N3 = output3.shape[1]
z0 = np.c_[output3, np.zeros(N3)]
color3 = LinearColorMapper(palette=palette, low=0, high=0.03)

plots[2].image(
    image=[z0],
    x=x1.min(),
    y=x2.min(),
    dw=(x1.max() - x1.min()) * (1 + 1/N3),
    dh=x2.max() - x2.min(),
    color_mapper=color3,
    alpha=0.8,
    level="underlay"
)

cb3 = ColorBar(
    color_mapper=color3,
    location=(0, 0),
    width=10,
    major_label_text_font_size="12pt",
    formatter=BasicTickFormatter(use_scientific=False, power_limit_low=-1)
)
plots[2].add_layout(cb3, 'right')

numerical_decision_boundary(plots[2], x1, x2, output3, level=0.0015, color='black')

for p in plots:
    p.output_backend = 'svg'

bokeh.io.show(bokeh.layouts.row(plots))

### Figure 5B3: homogeneous affinities with endogenous demand


In [14]:
from bokeh.models import LinearColorMapper, ColorBar, BasicTickFormatter

palette = bokeh.palettes.grey(15)[::-1]
t = np.linspace(0, 10, 100)

pm = TL_Competition()
pm.x3 = 1.5

N = 25
x1 = np.linspace(0, 1, N)
x2 = np.linspace(0, 1, N)

fig_size = [320, 225]
plots = []
for i in range(3):
    plots.append(
        bokeh.plotting.figure(
            width=fig_size[0],
            height=fig_size[1],
            x_range=bokeh.models.Range1d(x1[0], x1[-1]),
            y_range=bokeh.models.Range1d(x2[0], x2[-1])
        )
    )
    plots[i].axis.major_label_text_font_size = '13pt'

# ---------------- Run simulation ---------------- #
output1, output2, output3 = run_mlp(pm, x1, x2, N, t)

# ---------------- Node 1 ---------------- #
contourf(plots[0], x1, x2, output1, palette=palette, normal=False)
numerical_decision_boundary(plots[0], x1, x2, output1, level=0.05)

# ---------------- Node 2 ---------------- #
contourf(plots[1], x1, x2, output2, palette=palette, normal=False)
numerical_decision_boundary(plots[1], x1, x2, output2, level=0.05)

# ---------------- Output ---------------- #
N3 = output3.shape[1]
z0 = np.c_[output3, np.zeros(N3)]
color3 = LinearColorMapper(palette=palette, low=0, high=0.03)

plots[2].image(
    image=[z0],
    x=x1.min(),
    y=x2.min(),
    dw=(x1.max() - x1.min()) * (1 + 1/N3),
    dh=x2.max() - x2.min(),
    color_mapper=color3,
    alpha=0.8,
    level="underlay"
)

cb3 = ColorBar(
    color_mapper=color3,
    location=(0, 0),
    width=10,
    major_label_text_font_size="12pt",
    formatter=BasicTickFormatter(use_scientific=False, power_limit_low=-1)
)
plots[2].add_layout(cb3, 'right')

numerical_decision_boundary(plots[2], x1, x2, output3, level=0.0015, color='black')

for p in plots:
    p.output_backend = 'svg'

bokeh.io.show(bokeh.layouts.row(plots))

### Figure 6B1: heterogeneous affinities with endogenous demand


In [15]:
# ---------------- Simulation setup ---------------- #
palette = bokeh.palettes.grey(15)[::-1]
t = np.linspace(0, 10, 100)

K_base = 1.1375e-05
v_factor = 5

pm = TL_Competition()
pm.x3 = 1.5

# ---------------- Parameter variations ---------------- #
# Node 1: x1 favored
pm.a1  = (pm.d1  + pm.v1  + pm.d + pm.p) / (K_base / v_factor)  # Node 1, x1 → y1
pm.a2  = (pm.d2  + pm.v2  + pm.d + pm.p) / K_base               # Node 1, x2 → y2
pm.a4  = (pm.d4  + pm.v4  + pm.d + pm.p) / K_base               # Node 1, x0 → y2
pm.a3  = (pm.d3  + pm.v3  + pm.d + pm.p) / K_base               # Endogenous

# Node 2: x2 favored
pm.a5  = (pm.d5  + pm.v5  + pm.d + pm.p) / (K_base / v_factor)  # Node 2, x2 → y4
pm.a6  = (pm.d6  + pm.v6  + pm.d + pm.p) / K_base               # Node 2, x1 → y5
pm.a8  = (pm.d8  + pm.v8  + pm.d + pm.p) / K_base               # Node 2, x0 → y5

# Output: symmetric
pm.a9  = (pm.d9  + pm.v9  + pm.d + pm.p) / K_base               # Output, y1 → y6
pm.a11 = (pm.d11 + pm.v11 + pm.d + pm.p) / K_base               # Output, y4 → y6
pm.a10 = (pm.d10 + pm.v10 + pm.d + pm.p) / K_base               # Output, x0 → y7

# ---------------- Input grid ---------------- #
N = 25
x1 = np.linspace(0, 1, N)
x2 = np.linspace(0, 1, N)

fig_size = [320, 225]
plots = []
for i in range(3):
    plots.append(
        bokeh.plotting.figure(
            width=fig_size[0],
            height=fig_size[1],
            x_range=bokeh.models.Range1d(x1[0], x1[-1]),
            y_range=bokeh.models.Range1d(x2[0], x2[-1])
        )
    )
    plots[i].axis.major_label_text_font_size = '13pt'

# ---------------- Run simulation ---------------- #
output1, output2, output3 = run_mlp(pm, x1, x2, N, t)

# ---------------- Node 1 ---------------- #
contourf(plots[0], x1, x2, output1, palette=palette, normal=False)
numerical_decision_boundary(plots[0], x1, x2, output1, level=0.05)

# ---------------- Node 2 ---------------- #
contourf(plots[1], x1, x2, output2, palette=palette, normal=False)
numerical_decision_boundary(plots[1], x1, x2, output2, level=0.05)

# ---------------- Output ---------------- #
N3 = output3.shape[1]
z0 = np.c_[output3, np.zeros(N3)]
color3 = bokeh.models.LinearColorMapper(palette=palette, low=0, high=0.03)

plots[2].image(
    image=[z0],
    x=x1.min(),
    y=x2.min(),
    dw=(x1.max() - x1.min()) * (1 + 1/N3),
    dh=x2.max() - x2.min(),
    color_mapper=color3,
    alpha=0.8,
    level="underlay"
)

cb3 = bokeh.models.ColorBar(
    color_mapper=color3,
    location=(0, 0),
    width=10,
    major_label_text_font_size="12pt",
    formatter=bokeh.models.BasicTickFormatter(use_scientific=False, power_limit_low=-1)
)
plots[2].add_layout(cb3, 'right')

numerical_decision_boundary(plots[2], x1, x2, output3, level=0.0015, color='black')

# ---------------- Export ---------------- #
for p in plots:
    p.output_backend = 'svg'

bokeh.io.show(bokeh.layouts.row(plots))

### Figure 6B2: uncertainty under binding-affinity variation


In [16]:
# ---------------- Simulation setup ---------------- #
palette = bokeh.palettes.grey(15)[::-1]
t = np.linspace(0, 10, 100)

K_base = 1.1375e-05
v_factor = 5
vr = 0.5

N = 25
x1 = np.linspace(0, 1, N)
x2 = np.linspace(0, 1, N)

fig_size = [320, 225]
plots = []
for i in range(3):
    plots.append(
        bokeh.plotting.figure(
            width=fig_size[0],
            height=fig_size[1],
            x_range=bokeh.models.Range1d(x1[0], x1[-1]),
            y_range=bokeh.models.Range1d(x2[0], x2[-1])
        )
    )
    plots[i].axis.major_label_text_font_size = '13pt'

# ---------------- Helper to set affinities ---------------- #
def set_affinities(pm, vf):
    pm.a1  = (pm.d1  + pm.v1  + pm.d + pm.p) / (K_base / (v_factor * vf))  # Node 1, x1 → y1
    pm.a2  = (pm.d2  + pm.v2  + pm.d + pm.p) / K_base                       # Node 1, x2 → y2
    pm.a4  = (pm.d4  + pm.v4  + pm.d + pm.p) / K_base                       # Node 1, x0 → y2
    pm.a3  = (pm.d3  + pm.v3  + pm.d + pm.p) / K_base                       # Endogenous
    pm.a5  = (pm.d5  + pm.v5  + pm.d + pm.p) / (K_base / (v_factor * vf))  # Node 2, x2 → y4
    pm.a6  = (pm.d6  + pm.v6  + pm.d + pm.p) / K_base                       # Node 2, x1 → y5
    pm.a8  = (pm.d8  + pm.v8  + pm.d + pm.p) / K_base                       # Node 2, x0 → y5
    pm.a9  = (pm.d9  + pm.v9  + pm.d + pm.p) / K_base                       # Output, y1 → y6
    pm.a11 = (pm.d11 + pm.v11 + pm.d + pm.p) / K_base                       # Output, y4 → y6
    pm.a10 = (pm.d10 + pm.v10 + pm.d + pm.p) / K_base                       # Output, x0 → y7

# ---------------- Run nominal ---------------- #
pm = TL_Competition()
pm.x3 = 1.5
set_affinities(pm, vf=1.0)
output1, output2, output3 = run_mlp(pm, x1, x2, N, t)

# ---------------- Run -60% ---------------- #
pm_lo = TL_Competition()
pm_lo.x3 = 1.5
set_affinities(pm_lo, vf=1 - vr)
output1_lo, output2_lo, output3_lo = run_mlp(pm_lo, x1, x2, N, t)

# ---------------- Run +60% ---------------- #
pm_hi = TL_Competition()
pm_hi.x3 = 1.5
set_affinities(pm_hi, vf=1 + vr)
output1_hi, output2_hi, output3_hi = run_mlp(pm_hi, x1, x2, N, t)

# ---------------- Node 1 ---------------- #
contourf(plots[0], x1, x2, output1, palette=palette, normal=False)
numerical_decision_boundary(plots[0], x1, x2, output1,    level=0.05, color='black')
numerical_decision_boundary(plots[0], x1, x2, output1_lo, level=0.05, color='red')
numerical_decision_boundary(plots[0], x1, x2, output1_hi, level=0.05, color='blue')

# ---------------- Node 2 ---------------- #
contourf(plots[1], x1, x2, output2, palette=palette, normal=False)
numerical_decision_boundary(plots[1], x1, x2, output2,    level=0.05, color='black')
numerical_decision_boundary(plots[1], x1, x2, output2_lo, level=0.05, color='red')
numerical_decision_boundary(plots[1], x1, x2, output2_hi, level=0.05, color='blue')

# ---------------- Output ---------------- #
N3 = output3.shape[1]
z0 = np.c_[output3, np.zeros(N3)]
color3 = bokeh.models.LinearColorMapper(palette=palette, low=0, high=0.03)

plots[2].image(
    image=[z0],
    x=x1.min(), y=x2.min(),
    dw=(x1.max() - x1.min()) * (1 + 1/N3),
    dh=x2.max() - x2.min(),
    color_mapper=color3,
    alpha=0.8,
    level="underlay"
)

cb3 = bokeh.models.ColorBar(
    color_mapper=color3,
    location=(0, 0),
    width=10,
    major_label_text_font_size="12pt",
    formatter=bokeh.models.BasicTickFormatter(use_scientific=False, power_limit_low=-1)
)
plots[2].add_layout(cb3, 'right')

numerical_decision_boundary(plots[2], x1, x2, output3,    level=0.0015, color='black')
numerical_decision_boundary(plots[2], x1, x2, output3_lo, level=0.0015, color='red')
numerical_decision_boundary(plots[2], x1, x2, output3_hi, level=0.0015, color='blue')

# ---------------- Export ---------------- #
for p in plots:
    p.output_backend = 'svg'

bokeh.io.show(bokeh.layouts.row(plots))

#### Light/dark classification regions for the two endpoint settings


In [17]:
# Figure 6B2 — high-output regions and uncertainty
# Requires the notebook imports and the TL_Competition, Seq_TL_ODEs,
# and run_mlp definitions above; the preceding plotting cell is not needed.
# Light: above threshold in both sampled settings; dark: settings disagree.
# White: below threshold in both settings.
import contourpy

# ---------------- Simulation setup ---------------- #
t = np.linspace(0, 10, 100)

K_base = 1.1375e-05
v_factor = 5
vr = 0.5

N = 25
x1 = np.linspace(0, 1, N)
x2 = np.linspace(0, 1, N)

def set_affinities(pm, vf):
    pm.a1  = (pm.d1  + pm.v1  + pm.d + pm.p) / (K_base / (v_factor * vf))
    pm.a2  = (pm.d2  + pm.v2  + pm.d + pm.p) / K_base
    pm.a4  = (pm.d4  + pm.v4  + pm.d + pm.p) / K_base
    pm.a3  = (pm.d3  + pm.v3  + pm.d + pm.p) / K_base
    pm.a5  = (pm.d5  + pm.v5  + pm.d + pm.p) / (K_base / (v_factor * vf))
    pm.a6  = (pm.d6  + pm.v6  + pm.d + pm.p) / K_base
    pm.a8  = (pm.d8  + pm.v8  + pm.d + pm.p) / K_base
    pm.a9  = (pm.d9  + pm.v9  + pm.d + pm.p) / K_base
    pm.a11 = (pm.d11 + pm.v11 + pm.d + pm.p) / K_base
    pm.a10 = (pm.d10 + pm.v10 + pm.d + pm.p) / K_base

# ---------------- Run -50% and +50% association-rate settings ---------------- #
pm_lo = TL_Competition()
pm_lo.rtot = 3.0
pm_lo.x3 = 1.5
set_affinities(pm_lo, vf=1 - vr)
output1_lo, output2_lo, output3_lo = run_mlp(pm_lo, x1, x2, N, t)

pm_hi = TL_Competition()
pm_hi.rtot = 3.0
pm_hi.x3 = 1.5
set_affinities(pm_hi, vf=1 + vr)
output1_hi, output2_hi, output3_hi = run_mlp(pm_hi, x1, x2, N, t)

def _b2_fill_above(plot, field, threshold, color):
    """Fill a superlevel region, preserving disconnected polygons and holes."""
    field = np.asarray(field)
    upper = float(np.max(field))
    if upper <= threshold:
        return
    upper += max(abs(upper), abs(threshold), 1.0) * 1e-6
    generator = contourpy.contour_generator(
        x=x1, y=x2, z=field, fill_type="OuterOffset",
    )
    polygons, offsets = generator.filled(threshold, upper)
    polygons_x, polygons_y = [], []
    for vertices, ring_offsets in zip(polygons, offsets):
        rings = [
            vertices[start:stop]
            for start, stop in zip(ring_offsets[:-1], ring_offsets[1:])
        ]
        polygons_x.append([ring[:, 0].tolist() for ring in rings])
        polygons_y.append([ring[:, 1].tolist() for ring in rings])
    if polygons_x:
        plot.multi_polygons(
            xs=[polygons_x], ys=[polygons_y],
            fill_color=color, fill_alpha=1.0, line_color=None,
        )


def _b2_draw_boundary(plot, field, threshold):
    """Draw contour segments separately, without closing open boundaries."""
    generator = contourpy.contour_generator(
        x=x1, y=x2, z=field, line_type="Separate",
    )
    for segment in generator.lines(threshold):
        if len(segment) > 1:
            plot.line(
                segment[:, 0], segment[:, 1],
                line_color="black", line_width=2,
            )


b2_region_plots = []
_b2_nodes = [
    ("Node 1", output1_lo, output1_hi, 0.05, "#AEC0DC", "#E8EFF8"),
    ("Node 2", output2_lo, output2_hi, 0.05, "#EFBEA0", "#FBEDE3"),
    ("Node 3", output3_lo, output3_hi, 0.0015, "#B7B7B7", "#ECECEC"),
]
for title, output_lo, output_hi, threshold, band_color, light_color in _b2_nodes:
    plot = bokeh.plotting.figure(
        width=275, height=225,
        x_range=bokeh.models.Range1d(x1[0], x1[-1]),
        y_range=bokeh.models.Range1d(x2[0], x2[-1]),
        background_fill_color="white", output_backend="svg",
    )
    plot.axis.major_label_text_font_size = "13pt"

    # Fill where either output is high, then lighten where both are high.
    # This uses the scalar fields rather than assuming which side is active.
    _b2_fill_above(plot, np.maximum(output_lo, output_hi), threshold, band_color)
    _b2_fill_above(plot, np.minimum(output_lo, output_hi), threshold, light_color)
    _b2_draw_boundary(plot, output_lo, threshold)
    _b2_draw_boundary(plot, output_hi, threshold)
    b2_region_plots.append(plot)

bokeh.io.show(bokeh.layouts.row(*b2_region_plots))
